<a href="https://colab.research.google.com/github/Pranesh25/UnderstandingMachine/blob/main/Boston_house_prediction_with_FAST_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

california_housing = fetch_california_housing(as_frame=True)
X = california_housing.data
y = california_housing.target

print("California housing dataset loaded successfully.")
print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("\nFirst 5 rows of features (X):")
print(X.head())

California housing dataset loaded successfully.
Features (X) shape: (20640, 8)
Target (y) shape: (20640,)

First 5 rows of features (X):
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split into training and testing sets.")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Data split into training and testing sets.
X_train shape: (16512, 8)
X_test shape: (4128, 8)
y_train shape: (16512,)
y_test shape: (4128,)


In [4]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")
print("Model coefficients:", model.coef_)
print("Model intercept:", model.intercept_)

Linear Regression model trained successfully.
Model coefficients: [ 4.48674910e-01  9.72425752e-03 -1.23323343e-01  7.83144907e-01
 -2.02962058e-06 -3.52631849e-03 -4.19792487e-01 -4.33708065e-01]
Model intercept: -37.02327770606409


In [5]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Evaluation on Test Set:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Model Evaluation on Test Set:
Mean Squared Error (MSE): 0.56
R-squared (R2): 0.58


In [6]:
import joblib

# Save the trained model to a file
joblib.dump(model, 'linear_regression_model.pkl')

print("Model saved to linear_regression_model.pkl")

Model saved to linear_regression_model.pkl


Before we create the FastAPI application, we need to install `fastapi` and `uvicorn`. Uvicorn is an ASGI server that will run our FastAPI application.

In [7]:
!pip install fastapi uvicorn pydantic

Now, let's create the FastAPI application. We'll define a Pydantic model for input validation and a prediction endpoint. This code will be written to a file named `app.py`.

In [8]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd

# Initialize FastAPI app
app = FastAPI()

# Load the pre-trained model
model = joblib.load('linear_regression_model.pkl')

# Define a Pydantic model for input data validation
class HousingPredictInput(BaseModel):
    MedInc: float
    HouseAge: float
    AveRooms: float
    AveBedrms: float
    Population: float
    AveOccup: float
    Latitude: float
    Longitude: float

@app.get("/heartbeat")
def heartbeat():
    return {"status": "alive"}

# Define the prediction endpoint
@app.post("/predict")
def predict_housing_price(data: HousingPredictInput):
    # Convert input data to DataFrame for prediction
    input_df = pd.DataFrame([data.dict()])

    # Make prediction
    prediction = model.predict(input_df)[0]

    return {"predicted_price": prediction}



Writing app.py


To run this FastAPI application, you would typically execute the following command in your terminal. However, since we are in Colab, running it directly can be tricky. For demonstration purposes, you can consider running this locally or in a separate Colab instance or a dedicated virtual machine.

```bash
!uvicorn app:app --host 0.0.0.0 --port 8000
```

Once the server is running, you can access the interactive API documentation at `http://localhost:8000/docs` (or the Colab-provided public URL if using `ngrok` or similar tunneling services). You can then send POST requests to `http://localhost:8000/predict` with a JSON body matching the `HousingPredictInput` schema to get predictions.

To run your FastAPI application directly within Google Colab, we need to use `nest_asyncio` to handle the asynchronous event loop properly, and then call `uvicorn.run()` from within a Python cell. This will start your server directly in the notebook environment.

In [9]:
!pip install nest_asyncio

In [12]:
import nest_asyncio
import uvicorn
import threading
from app import app # Import the 'app' object from app.py

nest_asyncio.apply()

# Define a function to run the uvicorn server
def run_uvicorn_in_thread():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Create and start a thread for the uvicorn server
uvicorn_thread = threading.Thread(target=run_uvicorn_in_thread)
uvicorn_thread.start()

print("FastAPI application started in a separate thread.")
print("The server is running on http://0.0.0.0:8000.")
print("You can now proceed to set up ngrok to expose this service publicly.")

FastAPI application started in a separate thread.
The server is running on http://0.0.0.0:8000.
You can now proceed to set up ngrok to expose this service publicly.


After running the cell above, your FastAPI server will be active. To interact with it from your local machine, you'll need to set up a tunneling service like `ngrok`.

1.  **Get an `ngrok` Authtoken:** Sign up for an `ngrok` account and get your authtoken from the `ngrok` dashboard.
2.  **Install `pyngrok`:** `!pip install pyngrok`
3.  **Authenticate `ngrok`:** `!ngrok authtoken YOUR_AUTHTOKEN`
4.  **Start `ngrok` tunnel:** Run the following in a *new* cell to expose your FastAPI service:

    ```python
    from pyngrok import ngrok
    # Terminate any previous ngrok tunnels
    ngrok.kill()
    # Open a tunnel to port 8000
    public_url = ngrok.connect(8000)
    print(f"Public URL: {public_url}")
    ```

Once the `ngrok` tunnel is established, you can use the provided `public_url` to access your FastAPI endpoints from any browser or API client (e.g., Postman, `requests` in Python). For example, you can visit `public_url/docs` to see the OpenAPI documentation.

In [15]:
import os

# Install pyngrok
!pip install pyngrok -qq

# Get ngrok authtoken from secrets or user input
# It's recommended to store your ngrok authtoken in Colab Secrets (under the key icon on the left panel)

NGROK_AUTH_TOKEN = "24rFSTaNuQMx2w0wQNZAITHTDqN_tWNRmEnqW6DsjvkPn8jy"
try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    if NGROK_AUTH_TOKEN:
        print("ngrok authtoken retrieved from Colab Secrets.")
except Exception:
    pass # SecretNotFoundError or other exceptions

if not NGROK_AUTH_TOKEN:
    print("NGROK_AUTH_TOKEN not found in Colab Secrets or access failed. Please enter it manually.")
    NGROK_AUTH_TOKEN = input("Enter your ngrok authtoken: ")

# Authenticate ngrok
if NGROK_AUTH_TOKEN:
    !ngrok authtoken $NGROK_AUTH_TOKEN
    print("ngrok authentication complete.")
else:
    print("Warning: ngrok authtoken not provided. ngrok may not function correctly.")

<frozen posixpath>:82: RuntimeWarning: coroutine 'Server.serve' was never awaited


NGROK_AUTH_TOKEN not found in Colab Secrets or access failed. Please enter it manually.
Enter your ngrok authtoken: 24rFSTaNuQMx2w0wQNZAITHTDqN_tWNRmEnqW6DsjvkPn8jy
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
ngrok authentication complete.


In [16]:
from pyngrok import ngrok

# Terminate any previous ngrok tunnels to avoid conflicts
ngrok.kill()

# Open a tunnel to port 8000 where your FastAPI app is running
# The FastAPI server is running in a background thread and should be active.
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print(f"Access your FastAPI docs here: {public_url}/docs")
print(f"You can now send POST requests to {public_url}/predict")

Public URL: NgrokTunnel: "https://5d15-34-125-170-14.ngrok-free.app" -> "http://localhost:8000"
Access your FastAPI docs here: NgrokTunnel: "https://5d15-34-125-170-14.ngrok-free.app" -> "http://localhost:8000"/docs
You can now send POST requests to NgrokTunnel: "https://5d15-34-125-170-14.ngrok-free.app" -> "http://localhost:8000"/predict
